# This notebook will Tranform and join Invoice, Payment, Suplier data and prepare Analytics Ready data.

In [1]:
# Define Parameters
silver_catalog="demo_fusion_silver"
silver_schema="demo_fusion_silver_schema"
gold_catalog      = "demo_fusion_gold"
gold_schema       = "aidp"

In [2]:
df_inv_header = spark.read.table(f"{silver_catalog}.{silver_schema}.fusion_invoice_header")
df_inv_line = spark.read.table(f"{silver_catalog}.{silver_schema}.fusion_invoice_line")
df_pmt_hist = spark.read.table(f"{silver_catalog}.{silver_schema}.fusion_payment_history")
df_pmt_term = spark.read.table(f"{silver_catalog}.{silver_schema}.fusion_payment_term_header")
df_sup_master = spark.read.table(f"{silver_catalog}.{silver_schema}.fusion_external_supplier_master")


In [3]:
df_inv_header.printSchema()
df_inv_line.printSchema()

root
 |-- apinvoicesinvoiceid: string (nullable = true)
 |-- apinvoicesinvoicenum: string (nullable = true)
 |-- apinvoicesvendorid: string (nullable = true)
 |-- apinvoicesorgid: string (nullable = true)
 |-- apinvoicesinvoicedate: date (nullable = true)
 |-- apinvoicesinvoicereceiveddate: date (nullable = true)
 |-- apinvoicesinvoiceamount: double (nullable = true)
 |-- apinvoicesbaseamount: double (nullable = true)
 |-- apinvoicesinvoicecurrencycode: string (nullable = true)
 |-- apinvoicestermsid: string (nullable = true)
 |-- apinvoicespaymentstatusflag: string (nullable = true)
 |-- apinvoicesapprovalstatus: string (nullable = true)
 |-- apinvoicescreationdate: date (nullable = true)
 |-- apinvoiceslastupdatedate: date (nullable = true)

root
 |-- apinvoicelinesinvoiceid: string (nullable = true)
 |-- apinvoicelinesinvoicelineid: string (nullable = true)
 |-- apinvoicelineslineamount: double (nullable = true)
 |-- apinvoicelinesquantity: double (nullable = true)
 |-- apinvoicelin

In [4]:
# Join invoice header with invoice line

invoice_full_df = df_inv_header.join(
    df_inv_line,
    df_inv_header["APINVOICESINVOICEID"] == df_inv_line["APINVOICELINESINVOICEID"],
    how="left"
)

In [5]:
# Join payment history ---
invoice_payment_df = invoice_full_df.join(
    df_pmt_hist,
    df_inv_header["APINVOICESINVOICEID"] == df_pmt_hist["APPAYMENTHISTORYINVOICEID"],
    how="left"
)

In [6]:
# Join payment terms ---
invoice_payment_term_df = invoice_payment_df.join(
    df_pmt_term,
    df_inv_header["APINVOICESTERMSID"] == df_pmt_term["APPAYMENTTERMSID"],
    how="left"
)

In [7]:
# Join supplier master ---
final_df = invoice_payment_term_df.join(
    df_sup_master,
    df_inv_header["APINVOICESVENDORID"] == df_sup_master["SUPPLIER_ID"],
    how="left"
)

In [8]:
from pyspark.sql.functions import col, coalesce, lit, to_date, expr, datediff, when, count, avg, sum as _sum, current_timestamp

final_df = final_df.withColumn("INVOICE_DATE", to_date(col("APINVOICESINVOICEDATE"))) \
                   .withColumn("PAYMENT_DATE", to_date(col("APPAYMENTHISTORYPAYMENTDATE")))

final_df = final_df.withColumn("TERM_DAYS", lit(30))

final_df = final_df.withColumn(
    "DUE_DATE",
    expr("date_add(INVOICE_DATE, TERM_DAYS)")
)

final_df = final_df.withColumn("DAYS_TO_PAY", datediff(col("PAYMENT_DATE"), col("INVOICE_DATE"))) \
                   .withColumn("PAYMENT_DELAY", datediff(col("PAYMENT_DATE"), col("DUE_DATE"))) \
                   .withColumn("LATE_PAYMENT_FLAG", when(col("PAYMENT_DELAY") > 0, lit(1)).otherwise(lit(0)))

final_df = final_df \
    .withColumn("SOURCE_SYSTEM", lit("FUSION")) \
    .withColumn("LOAD_TIMESTAMP", current_timestamp())

supplier_kpi_df = final_df.groupBy("SUPPLIER_ID").agg(
    count("APINVOICESINVOICEID").alias("INVOICE_COUNT"),
    _sum("APINVOICESINVOICEAMOUNT").alias("TOTAL_INVOICE_AMOUNT"),
    avg("DAYS_TO_PAY").alias("AVG_DAYS_TO_PAY"),
    avg("PAYMENT_DELAY").alias("AVG_PAYMENT_DELAY"),
    avg("LATE_PAYMENT_FLAG").alias("LATE_PAYMENT_RATIO")
)

In [9]:
supplier_kpi_df = (
    final_df.groupBy("SUPPLIER_ID")
    .agg(
        count("APINVOICESINVOICEID").alias("INVOICE_COUNT"),
        _sum("APINVOICESINVOICEAMOUNT").alias("TOTAL_INVOICE_AMOUNT"),
        avg("DAYS_TO_PAY").alias("AVG_DAYS_TO_PAY"),
        avg("PAYMENT_DELAY").alias("AVG_PAYMENT_DELAY"),
        avg("LATE_PAYMENT_FLAG").alias("LATE_PAYMENT_RATIO")
    )
)

In [10]:
supplier_kpi_df = supplier_kpi_df.withColumn("LOAD_TIMESTAMP", current_timestamp())

# Select columns in the exact same order as Oracle DDL
supplier_kpi_df = supplier_kpi_df.select(
    "SUPPLIER_ID",
    "INVOICE_COUNT",
    "TOTAL_INVOICE_AMOUNT",
    "AVG_DAYS_TO_PAY",
    "AVG_PAYMENT_DELAY",
    "LATE_PAYMENT_RATIO",
    "LOAD_TIMESTAMP"
)

In [11]:
fusion_invoice_supplier_df = final_df.select(
    "APINVOICESINVOICEID",
    "APINVOICESINVOICEDATE",
    "APPAYMENTHISTORYPAYMENTDATE",
    "TERM_DAYS",
    "DUE_DATE",
    "DAYS_TO_PAY",
    "PAYMENT_DELAY",
    "LATE_PAYMENT_FLAG",
    "APINVOICESVENDORID",
    "SUPPLIER_ID",
    "APINVOICESINVOICEAMOUNT",
    "APINVOICESBASEAMOUNT",
    "APINVOICESINVOICECURRENCYCODE",
    "APINVOICESPAYMENTSTATUSFLAG",
    "APINVOICESAPPROVALSTATUS",
    "APINVOICESORGID",
    "INVOICE_DATE",
    "PAYMENT_DATE",
    "SOURCE_SYSTEM",
    "LOAD_TIMESTAMP"
)

In [12]:
fusion_invoice_supplier_df.write.insertInto(f"{gold_catalog}.{gold_schema}.fusion_invoice_supplier")


In [13]:
supplier_kpi_df.select("SUPPLIER_ID").distinct().show(20)

+---------------+
|    SUPPLIER_ID|
+---------------+
|300000047414571|
|300000047414503|
|300000047572113|
|300000047414679|
|300000047414635|
|300000047507596|
|           NULL|
+---------------+



In [14]:
supplier_kpi_df = supplier_kpi_df.filter(col("SUPPLIER_ID").isNotNull())
supplier_kpi_df.write.insertInto(f"{gold_catalog}.{gold_schema}.fusion_supplier_kpi")